In [ ]:
%matplotlib inline

from pathlib import Path
import sys
import os
import json
import torch
import pandas as pd

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT))

import vix_tcn_revin_xai_plus as vtrx

CSV_PATH = ROOT / "data" / "raw" / "timeseries_data.csv"
OUT_DIR = ROOT / "outputs"
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE)

In [ ]:
cfg = vtrx.Config(
    csv_path=str(CSV_PATH),
    index_col="날짜",
    drop_cols=("Silver", "Copper", "USD/GBP", "USD/CNY", "USD/JPY", "USD/EUR", "USD/CAD"),
    target_col="VIX",
    deterministic=False,
    use_amp=True,
    num_workers=0,
    pin_memory=(DEVICE.type == "cuda"),
    epochs=300,
    patience=20,
    min_epoch=50,
    param_budget=4000,
    revin_affine=True,
    seq_len=20,
    batch_size=64,
    out_dir=str(OUT_DIR),
)

cfg

In [ ]:
df_raw = vtrx.load_frame(cfg.csv_path, cfg.index_col, list(cfg.drop_cols))

print(f"shape      : {df_raw.shape}")
print(f"date range : {df_raw.index[0]} ~ {df_raw.index[-1]}")
print(f"columns    : {list(df_raw.columns)}")
df_raw.head()

In [ ]:
RUN_SUITE = True
ARCHITECTURES = ["tcn", "cnn"]
SEEDS = (5,)

In [ ]:
if RUN_SUITE:
    suite = vtrx.run_experiment_suite(
        cfg=cfg,
        df_raw=df_raw,
        architectures=ARCHITECTURES,
        seeds=SEEDS,
        device=DEVICE,
    )
else:
    raise RuntimeError("RUN_SUITE=False")

In [ ]:
print("best_key         :", suite["best_key"])
print("best_val_rmse    :", suite["best_val_rmse"])
print("best_test_rmse   :", suite["best_test_rmse"])
print("best_tcn_key     :", suite["best_tcn_key"])
print("best_tcn_test    :", suite["best_tcn_test_rmse"])

suite["summary_df"]

In [ ]:
with open(exp_dir / "final_test_metrics.json", "r", encoding="utf-8") as f:
    final_metrics = json.load(f)

final_metrics

In [ ]:
pred_df = pd.read_csv(exp_dir / "predictions_best_test.csv")
pred_df.head()